In [ ]:
# Libraries
import networkx as nx
import pandas as pd
import numpy as np
import torch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Reading the filtered dataset
dataset = pd.read_csv('data/Mastodon_instances_edges_Sept2024_filtered.csv')
dataset.head()

In [ ]:
df_info = pd.read_csv('data/Mastodon_instances_metadata_v3.csv')
df_info.head()

# Get the list of nodes from df_info
valid_nodes = set(df_info['instance'].values)

print(len(valid_nodes))

# Filter the dataset to only include edges where both nodes are in valid_nodes
dataset = dataset[
    dataset['source_domain'].isin(valid_nodes) & 
    dataset['target_domain'].isin(valid_nodes)
]

print(f"Dataset shape after filtering: {dataset.shape}")

In [ ]:
# Remove self-loops (where source and target are the same)
dataset = dataset[dataset['source_domain'] != dataset['target_domain']]
print(f"Dataset shape after removing self-loops: {dataset.shape}")

In [ ]:
dataset.head()

In [ ]:
# Creating the graph from the dataset
graph = nx.from_pandas_edgelist(dataset, 'source_domain', 'target_domain', edge_attr='weight')

In [ ]:
# Create a mapping: name → index
node_to_idx = {node: i for i, node in enumerate(graph.nodes())}

# Reverse mapping: index → name
idx_to_node = {i: node for node, i in node_to_idx.items()}

In [ ]:
graph.edges

In [ ]:
from torch_geometric.utils import from_networkx

# Creating the graph from the dataset
data = from_networkx(graph)

In [ ]:
data

In [ ]:
print(data.edge_index)

In [ ]:
from torch_geometric.nn import Node2Vec
import torch

# Model configuration
model = Node2Vec(
    data.edge_index,
    embedding_dim=384,
    walk_length=20,
    context_size=10,
    walks_per_node=15,
    num_negative_samples=10,
    p=0.8,
    q=1.0,
    sparse=True,
).to(device)

In [ ]:
# Model configuration
loader = model.loader(batch_size=32, shuffle=True)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

In [ ]:
# Model training
def train():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in loader:
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

for epoch in range(1, 101):
    loss = train()
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}')

In [ ]:
import torch

# Save embeddings separately
embeddings = model.embedding.weight.detach().cpu().tolist()
torch.save(embeddings, "embeddings/node2vec_emb.pt")

print("Embeddings saved successfully!")

In [ ]:
import os
import pickle

embedding_cache_path = "embeddings/node2vec_emb.pkl"

embedding_dict = {}

# Creating the embedding dictionary
for node in graph.nodes:
    if node in node_to_idx:
        target_idx = node_to_idx[node]
    else:
        raise ValueError(f"Node {node} not found in the graph!")
    embedding_dict[node] = embeddings[target_idx]

In [ ]:
print(list(embedding_dict.items())[:5])

In [ ]:
# Saving the embedding dictionary
torch.save(embedding_dict, embedding_cache_path)